In [1]:
import sys
sys.path.insert(0, "..")

import geopandas as gpd

gdf = gpd.read_file("../data/processed/madrid_callejero_filtered.geojson")
print(gdf.shape)
print(gdf.columns.tolist())
print(gdf.crs)
gdf.head(3)

(61852, 24)
['u', 'v', 'key', 'osmid', 'highway', 'lanes', 'maxspeed', 'name', 'oneway', 'reversed', 'length', 'junction', 'bridge', 'width', 'access', 'ref', 'tunnel', 'est_width', 'zona', 'cerca_bomberos', 'cerca_hospitales', 'cerca_centros_educativos', 'cerca_centros_mayores', 'geometry']
EPSG:4326


c:\Users\Lenovo\miniconda3\envs\tfmp2\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


,u,v,key,osmid,highway,lanes,maxspeed,name,oneway,reversed,...,access,ref,tunnel,est_width,zona,cerca_bomberos,cerca_hospitales,cerca_centros_educativos,cerca_centros_mayores,geometry
0,171946,26513145,0,807334397,[secondary],[4],[50],[Calle de Velázquez],True,False,...,None,None,None,None,Salamanca,False,False,False,False,"LINESTRING (-3.68444 40.42125, -3.68444 40.421..."
1,171951,1209331009,0,104864843,[residential],None,None,[Calle Juan de Mena],True,False,...,None,None,None,None,Retiro,False,False,False,False,"LINESTRING (-3.68899 40.41736, -3.68885 40.41749)"
2,171951,26486636,0,553113575,[secondary],[3],[50],[Calle de Alfonso XII],True,False,...,None,None,None,None,Retiro,False,False,False,False,"LINESTRING (-3.68899 40.41736, -3.68903 40.416..."


In [2]:
for col in ["width", "est_width", "bridge", "tunnel"]:
    serie = gdf[col]
    print(f"{col} — nulos: {serie.isna().sum()}/{len(serie)}")
    print(f"  tipos encontrados: {serie.dropna().apply(type).value_counts().to_dict()}")
    ejemplos = serie.dropna().astype(str).unique()[:10]
    print(f"  ejemplos: {ejemplos}")
    print()

width — nulos: 61395/61852
  tipos encontrados: {<class 'numpy.ndarray'>: 457}
  ejemplos: <StringArray>
['['5.9']', '['2.5']',   '['6']', '['4.7']',   '['5']',   '['3']', '['3.3']',
   '['4']',   '['9']',   '['7']']
Length: 10, dtype: str

est_width — nulos: 61837/61852
  tipos encontrados: {<class 'numpy.ndarray'>: 15}
  ejemplos: <StringArray>
['['20']', '["['20' '10']"]', '['10']']
Length: 3, dtype: str

bridge — nulos: 61064/61852
  tipos encontrados: {<class 'numpy.ndarray'>: 788}
  ejemplos: <StringArray>
['['yes']', '['viaduct']', '["['yes' 'viaduct']"]']
Length: 3, dtype: str

tunnel — nulos: 61263/61852
  tipos encontrados: {<class 'numpy.ndarray'>: 589}
  ejemplos: <StringArray>
['['yes']', '["['yes' 'covered']"]', '['covered']', '['building_passage']']
Length: 4, dtype: str



In [3]:
import numpy as np
import re
import pandas as pd
from pipeline.transform.build_graph import DEFAULT_WIDTH_BY_HIGHWAY, DEFAULT_SPEED_BY_HIGHWAY

def _extraer_valor(val):
    """Extrae un único valor numérico de una celda que puede ser None, lista o string.
    Si hay varios valores en la lista, se queda con el primero (documentado, no oculto)."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    if isinstance(val, (list, np.ndarray)):
        if len(val) == 0:
            return None
        val = val[0]
    limpio = re.sub(r"[^\d.]", "", str(val))
    try:
        return float(limpio) if limpio else None
    except ValueError:
        return None

def _highway_norm(row):
    hw = row.get("highway", "unclassified")
    return hw[0] if isinstance(hw, (list, np.ndarray)) else hw

def clasificar_width(row):
    hw = _highway_norm(row)
    w_real = _extraer_valor(row.get("width"))
    if w_real is not None:
        return hw, w_real, "real_osm"
    w_est = _extraer_valor(row.get("est_width"))
    if w_est is not None:
        return hw, w_est, "estimado_osm"
    return hw, DEFAULT_WIDTH_BY_HIGHWAY.get(hw, 3.5), "estimado_propio"

def clasificar_speed(row):
    hw = _highway_norm(row)
    v_real = _extraer_valor(row.get("maxspeed"))
    if v_real is not None:
        return hw, v_real, "real_osm"
    return hw, DEFAULT_SPEED_BY_HIGHWAY.get(hw, 30.0), "estimado_propio"

width_info = gdf.apply(clasificar_width, axis=1, result_type="expand")
width_info.columns = ["highway", "width_m", "fuente_width"]

speed_info = gdf.apply(clasificar_speed, axis=1, result_type="expand")
speed_info.columns = ["highway", "speed_kph", "fuente_speed"]

print("=== COBERTURA DE ANCHURA por tipo de vía (%) ===")
tabla_width = (width_info.groupby("highway")["fuente_width"]
               .value_counts(normalize=True).unstack().fillna(0) * 100)
print(tabla_width.round(1))

print("\n=== COBERTURA DE VELOCIDAD por tipo de vía (%) ===")
tabla_speed = (speed_info.groupby("highway")["fuente_speed"]
               .value_counts(normalize=True).unstack().fillna(0) * 100)
print(tabla_speed.round(1))

=== COBERTURA DE ANCHURA por tipo de vía (%) ===
fuente_width                                estimado_osm  estimado_propio  \
highway                                                                     
['motorway' 'motorway_link']                         0.0            100.0   
['motorway' 'trunk']                                 0.0            100.0   
['motorway_link' 'primary' 'primary_link']           0.0            100.0   
['motorway_link' 'primary_link']                     0.0            100.0   
['motorway_link' 'unclassified']                     0.0            100.0   
['primary' 'motorway']                               0.0            100.0   
['primary' 'motorway_link']                          0.0            100.0   
['primary' 'primary_link']                           0.0            100.0   
['primary' 'secondary']                              0.0            100.0   
['primary_link' 'motorway_link']                     0.0            100.0   
['residential' 'living_stre

In [4]:
conteo_highway = gdf["highway"].apply(lambda h: str(h)).value_counts()
print(conteo_highway)

highway
['residential']                                   42083
['tertiary']                                       7656
['secondary']                                      3608
['primary']                                        3224
['living_street']                                  1442
['motorway_link']                                  1324
['unclassified']                                    889
['motorway']                                        691
['trunk']                                           247
['primary_link']                                    132
['secondary_link']                                  118
['tertiary_link']                                   112
['trunk_link']                                       94
['busway']                                           53
["['residential' 'living_street']"]                  47
["['motorway' 'motorway_link']"]                     20
["['residential' 'unclassified']"]                   20
["['residential' 'tertiary']"]          

In [5]:
def _highway_norm(row):
    hw = row.get("highway", "unclassified")
    if isinstance(hw, (list, np.ndarray)):
        hw = hw[0] if len(hw) > 0 else "unclassified"
    hw = str(hw)
    if hw.strip().startswith("["):
        primero = re.findall(r"[A-Za-z_]+", hw)
        hw = primero[0] if primero else "unclassified"
    return hw

In [6]:
width_info = gdf.apply(clasificar_width, axis=1, result_type="expand")
width_info.columns = ["highway", "width_m", "fuente_width"]

speed_info = gdf.apply(clasificar_speed, axis=1, result_type="expand")
speed_info.columns = ["highway", "speed_kph", "fuente_speed"]

print("=== COBERTURA DE ANCHURA por tipo de vía (%) ===")
tabla_width = (width_info.groupby("highway")["fuente_width"]
               .value_counts(normalize=True).unstack().fillna(0) * 100)
print(tabla_width.round(1))

print("\n=== COBERTURA DE VELOCIDAD por tipo de vía (%) ===")
tabla_speed = (speed_info.groupby("highway")["fuente_speed"]
               .value_counts(normalize=True).unstack().fillna(0) * 100)
print(tabla_speed.round(1))

=== COBERTURA DE ANCHURA por tipo de vía (%) ===
fuente_width    estimado_osm  estimado_propio  real_osm
highway                                                
busway                   0.0            100.0       0.0
living_street            0.0             98.5       1.5
motorway                 0.0             99.9       0.1
motorway_link            0.0            100.0       0.0
primary                  0.5             99.5       0.0
primary_link             0.0            100.0       0.0
residential              0.0             99.1       0.9
secondary                0.0             99.7       0.3
secondary_link           0.0            100.0       0.0
tertiary                 0.0             99.6       0.4
tertiary_link            0.0            100.0       0.0
trunk                    0.0            100.0       0.0
trunk_link               0.0            100.0       0.0
unclassified             0.0            100.0       0.0

=== COBERTURA DE VELOCIDAD por tipo de vía (%) ===
fue

In [7]:
n_con_corchetes = width_info["highway"].apply(lambda h: "[" in h).sum()
print(f"Categorías que siguen con corchetes: {n_con_corchetes}")

print("\nRecuento final por tipo de vía (ya normalizado):")
print(width_info["highway"].value_counts())

Categorías que siguen con corchetes: 0

Recuento final por tipo de vía (ya normalizado):
highway
residential       42178
tertiary           7672
secondary          3613
primary            3233
living_street      1442
motorway_link      1331
unclassified        895
motorway            714
trunk               257
primary_link        134
secondary_link      120
tertiary_link       113
trunk_link           97
busway               53
Name: count, dtype: int64


In [8]:
def _extraer_categoria(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None
    if isinstance(val, (list, np.ndarray)):
        if len(val) == 0:
            return None
        val = val[0]
    val = str(val).strip()
    if val.startswith("["):
        m = re.findall(r"[A-Za-z_]+", val)
        return m[0] if m else None
    return val

In [9]:
ALTURA_LIBRE_SIN_RESTRICCION = 99.0  # gálibo libre: no hay restricción real de altura
ALTURA_TUNEL_CONSERVADORA = 3.5      # asunción conservadora si no hay maxheight real

def clasificar_galibo(row):
    h_real = _extraer_valor(row.get("maxheight"))
    if h_real is not None:
        return h_real, "real"
    tunnel_val = _extraer_categoria(row.get("tunnel"))
    if tunnel_val is not None:
        return ALTURA_TUNEL_CONSERVADORA, f"estimado_tunel_{tunnel_val}"
    return ALTURA_LIBRE_SIN_RESTRICCION, "libre_sin_restriccion"

galibo_info = gdf.apply(clasificar_galibo, axis=1, result_type="expand")
galibo_info.columns = ["height_m", "fuente_galibo"]

print("=== COBERTURA DE GÁLIBO ===")
print(galibo_info["fuente_galibo"].value_counts())
print(f"\nTotal aristas con restricción real o estimada: {(galibo_info['fuente_galibo'] != 'libre_sin_restriccion').sum()} / {len(gdf)}")

=== COBERTURA DE GÁLIBO ===
fuente_galibo
libre_sin_restriccion              61263
estimado_tunel_yes                   541
estimado_tunel_building_passage       42
estimado_tunel_covered                 6
Name: count, dtype: int64

Total aristas con restricción real o estimada: 589 / 61852


In [10]:
from pipeline.transform.build_graph import build_graph

gdf_utm = gdf.to_crs(epsg=25830)
G = build_graph(gdf_utm)

print(f"Nodos: {G.number_of_nodes():,}")
print(f"Aristas: {G.number_of_edges():,}")

Nodos: 171,356
Aristas: 235,012


In [11]:
import collections

alturas = [d["height_m"] for _, _, d in G.edges(data=True)]
print(collections.Counter(alturas))

Counter({99.0: 228364, 3.5: 6375, 3.0: 273})


In [12]:
from pipeline.ingest.open_data_madrid import download_equipamientos

gdf_bomberos = download_equipamientos("bomberos")
gdf_hospitales = download_equipamientos("hospitales")

print(f"Parques de bomberos: {len(gdf_bomberos)}")
print(gdf_bomberos.head(3))
print(f"\nHospitales: {len(gdf_hospitales)}")
print(gdf_hospitales.head(3))

Parques de bomberos: 13
                             nombre  \
0   Parque de Bomberos 01. Chamberí   
1  Parque de Bomberos 02. Salamanca   
2     Parque de Bomberos 03. Centro   

                                                tipo  \
0  https://datos.madrid.es/egob/kos/entidadesYorg...   
1  https://datos.madrid.es/egob/kos/entidadesYorg...   
2  https://datos.madrid.es/egob/kos/entidadesYorg...   

                  direccion                   geometry  
0  CALLE SANTA ENGRACIA 118  POINT (-3.70082 40.44022)  
1     CALLE RUFINO BLANCO 2  POINT (-3.66711 40.42851)  
2          RONDA SEGOVIA 95  POINT (-3.71228 40.40698)  

Hospitales: 277
                                              nombre  \
0  Centro Concertado de adicciones (CCAD) Centro ...   
1  Centro Concertado de Atención a las Adicciones...   
2  Centro Concertado de Atención a las Adicciones...   

                                                tipo  \
0  https://datos.madrid.es/egob/kos/entidadesYorg...   
1  https://d

In [13]:
print(gdf_hospitales["tipo"].value_counts())

tipo
https://datos.madrid.es/egob/kos/entidadesYorganismos/CentrosAtencionMedica/CentrosSalud                          130
https://datos.madrid.es/egob/kos/entidadesYorganismos/CentrosAtencionMedica/HospitalesClinicasSanatorios           53
https://datos.madrid.es/egob/kos/entidadesYorganismos/CentrosAtencionMedica/CentrosPrevencionEnfermedades          26
https://datos.madrid.es/egob/kos/entidadesYorganismos/CentrosAtencionMedica/CentrosSaludMental                     26
https://datos.madrid.es/egob/kos/entidadesYorganismos/CentrosAtencionMedica/CentrosEspecialidadesMedicas           19
https://datos.madrid.es/egob/kos/entidadesYorganismos/CentrosAtencionMedica/CentrosAsistenciaDrogodependientes     18
https://datos.madrid.es/egob/kos/entidadesYorganismos/CentrosAtencionMedica/OtrosCentrosMedicos                     4
https://datos.madrid.es/egob/kos/entidadesYorganismos/CentrosAtencionSocial/CentrosRehabilitacionPsicosocial        1
Name: count, dtype: int64


In [14]:
gdf_hospitales_reales = gdf_hospitales[
    gdf_hospitales["tipo"].str.contains("HospitalesClinicasSanatorios", na=False)
].reset_index(drop=True)

print(f"Hospitales/clínicas/sanatorios reales: {len(gdf_hospitales_reales)}")
print(gdf_hospitales_reales[["nombre", "direccion"]].head(10))

Hospitales/clínicas/sanatorios reales: 53
                                              nombre  \
0                                     Clínica CEMTRO   
1        Clínica Doctor León. Maria Auxiliadora S.A.   
2                                    Clínica Isadora   
3                                 Clínica López Ibor   
4                   Clínica Nuestra Señora de la Paz   
5                                 Clínica San Miguel   
6           Clínica Universidad de Navarra en Madrid   
7                       Fundación Instituto San José   
8                          Fundación Vianorte-Laguna   
9  Hospital Beata María Ana de Hermanas Hospitala...   

                                          direccion  
0              AVENIDA VENTISQUERO DE LA CONDESA 42  
1                          PLAZA MARIANO DE CAVIA 3  
2                                  CALLE PIRINEOS 7  
3               CALLE DOCTOR JUAN JOSE LOPEZ IBOR 2  
4                          CALLE LOPEZ DE HOYOS 259  
5                

In [15]:
from pipeline.transform.build_graph import build_kdtree, add_special_nodes

# Reproyectamos los equipamientos al mismo CRS que el grafo (metros, no grados)
gdf_bomberos_utm = gdf_bomberos.to_crs(epsg=25830)
gdf_hospitales_utm = gdf_hospitales_reales.to_crs(epsg=25830)

kdtree, node_items = build_kdtree(G)

G = add_special_nodes(G, gdf_bomberos_utm, tipo_nodo="parque_bomberos", kdtree=kdtree, node_items=node_items, prefijo="bombero")
G = add_special_nodes(G, gdf_hospitales_utm, tipo_nodo="hospital", kdtree=kdtree, node_items=node_items, prefijo="hospital")

print(f"Nodos totales: {G.number_of_nodes():,}")
print(f"Aristas totales: {G.number_of_edges():,}")

nodos_bomberos = [n for n, d in G.nodes(data=True) if d.get("tipo_nodo") == "parque_bomberos"]
print(f"Nodos de bomberos añadidos: {len(nodos_bomberos)}")

Nodos totales: 171,422
Aristas totales: 235,144
Nodos de bomberos añadidos: 13


In [17]:
import networkx as nx

nodos_bomberos = [n for n, d in G.nodes(data=True) if d.get("tipo_nodo") == "parque_bomberos"]
total_nodos = G.number_of_nodes()

resultados = []
for nodo in nodos_bomberos:
    nombre = G.nodes[nodo]["nombre"]
    alcanzables = len(nx.descendants(G, nodo)) + 1
    resultados.append({
        "parque": nombre,
        "nodos_alcanzables": alcanzables,
        "porcentaje_red": round(alcanzables / total_nodos * 100, 1),
    })

import pandas as pd
df_conectividad = pd.DataFrame(resultados)
print(df_conectividad)

                                          parque  nodos_alcanzables  \
0                Parque de Bomberos 01. Chamberí             169151   
1               Parque de Bomberos 02. Salamanca             169151   
2                  Parque de Bomberos 03. Centro             169151   
3                  Parque de Bomberos 04. Tetuán             169151   
4                   Parque de Bomberos 05. Usera             169151   
5                  Parque de Bomberos 06. Centro             169151   
6                Parque de Bomberos 07. San Blas             169151   
7      Parque de Bomberos 08. Puente de Vallecas             169151   
8   Parque de Bomberos 09. Fuencarral - El Pardo             169151   
9              Parque de Bomberos 10. Villaverde             169151   
10              Parque de Bomberos 11. Hortaleza             169151   
11                 Parque de Bomberos 12. Latina             169151   
12              Parque de Bomberos 13. Vicálvaro             169151   

    p

In [18]:
componentes = list(nx.weakly_connected_components(G))
tamanos = sorted([len(c) for c in componentes], reverse=True)

print(f"Número total de componentes (islas): {len(componentes)}")
print(f"Tamaño de las 5 islas más grandes: {tamanos[:5]}")
print(f"Cuántas islas son diminutas (menos de 20 nodos): {sum(1 for t in tamanos if t < 20)}")

Número total de componentes (islas): 1
Tamaño de las 5 islas más grandes: [171422]
Cuántas islas son diminutas (menos de 20 nodos): 0


In [19]:
nodos_hospital = [n for n, d in G.nodes(data=True) if d.get("tipo_nodo") == "hospital"]
alcanzables_desde_bomberos_1 = nx.descendants(G, nodos_bomberos[0])

hospitales_inalcanzables = [
    G.nodes[n]["nombre"] for n in nodos_hospital if n not in alcanzables_desde_bomberos_1
]
print(f"Hospitales inalcanzables desde un parque de bomberos: {len(hospitales_inalcanzables)} / {len(nodos_hospital)}")
print(hospitales_inalcanzables)

Hospitales inalcanzables desde un parque de bomberos: 0 / 53
[]


In [21]:
# Nos quedamos solo con los inalcanzables que SÍ son cruces reales (más de 1 calle tocándolos)
cruces_reales_inalcanzables = [
    n for n in inalcanzables
    if (G.in_degree(n) + G.out_degree(n)) > 2
]
print(f"Cruces reales (no puntos de curva) inalcanzables: {len(cruces_reales_inalcanzables)} / {len(inalcanzables)}")

# Dónde están geográficamente (coordenadas en metros, EPSG:25830)
xs = [G.nodes[n]["x"] for n in inalcanzables]
ys = [G.nodes[n]["y"] for n in inalcanzables]
print(f"\nRango X de los inalcanzables: {min(xs):.0f} a {max(xs):.0f}")
print(f"Rango Y de los inalcanzables: {min(ys):.0f} a {max(ys):.0f}")

xs_todos = [d["x"] for _, d in G.nodes(data=True)]
ys_todos = [d["y"] for _, d in G.nodes(data=True)]
print(f"\nRango X de todo el grafo: {min(xs_todos):.0f} a {max(xs_todos):.0f}")
print(f"Rango Y de todo el grafo: {min(ys_todos):.0f} a {max(ys_todos):.0f}")

Cruces reales (no puntos de curva) inalcanzables: 133 / 2271

Rango X de los inalcanzables: 429332 a 455822
Rango Y de los inalcanzables: 4463582 a 4492384

Rango X de todo el grafo: 429051 a 455957
Rango Y de todo el grafo: 4463582 a 4492859


In [22]:
subG = G.subgraph(cruces_reales_inalcanzables)
grupos = list(nx.weakly_connected_components(subG))
tamanos_grupos = sorted([len(g) for g in grupos], reverse=True)

print(f"Número de grupos entre los 133 cruces problemáticos: {len(grupos)}")
print(f"Tamaños de los grupos más grandes: {tamanos_grupos[:15]}")
print(f"Cuántos son cruces completamente solos (grupo de tamaño 1): {sum(1 for t in tamanos_grupos if t == 1)}")

Número de grupos entre los 133 cruces problemáticos: 64
Tamaños de los grupos más grandes: [28, 12, 10, 6, 6, 4, 3, 3, 3, 2, 2, 2, 1, 1, 1]
Cuántos son cruces completamente solos (grupo de tamaño 1): 52


In [24]:
import pyproj
transformer = pyproj.Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)

detalles = []
for n in cruces_reales_inalcanzables:
    d = G.nodes[n]
    lon, lat = transformer.transform(d["x"], d["y"])
    detalles.append({
        "nodo": n, "lat": round(lat, 6), "lon": round(lon, 6),
        "entradas": G.in_degree(n), "salidas": G.out_degree(n),
    })

df_ubicaciones = pd.DataFrame(detalles)
print(df_ubicaciones.to_string())

grupo_grande = max(grupos, key=len)
detalles_grupo_grande = [d for d in detalles if d["nodo"] in grupo_grande]
print(f"\nEl grupo de {len(grupo_grande)} nodos está aquí:")
for d in detalles_grupo_grande[:5]:
    print(f"  https://www.google.com/maps?q={d['lat']},{d['lon']}")

       nodo        lat       lon  entradas  salidas
0     81994  40.386805 -3.753090         1        2
1     81995  40.386961 -3.753555         3        3
2     90435  40.387270 -3.699890         2        2
3     90436  40.387457 -3.699613         3        3
4     90437  40.387003 -3.700300         1        2
5     90439  40.387577 -3.699755         2        2
6    156095  40.373786 -3.804423         1        2
7      8838  40.503118 -3.659541         1        2
8     49956  40.432923 -3.700504         1        2
9     49963  40.432915 -3.700467         2        2
10    49964  40.432918 -3.700436         2        2
11   149966  40.413546 -3.522135         1        2
12   149970  40.413744 -3.522283         2        2
13   149971  40.413863 -3.522395         2        2
14   115568  40.397022 -3.563156         1        2
15   115580  40.401407 -3.570267         1        2
16   149974  40.414249 -3.522917         2        2
17   149975  40.414766 -3.523728         2        2
18   149976 

In [25]:
import pickle, json, collections, os
import networkx as nx

# --- 1. Serializar el grafo final ---
GRAFO_OUT = "../data/processed/grafo_madrid.pkl"
with open(GRAFO_OUT, "wb") as f:
    pickle.dump(G, f)
print(f"Grafo guardado: {GRAFO_OUT} ({os.path.getsize(GRAFO_OUT)/1024/1024:.1f} MB)")

# --- 2. Recalcular estadísticas clave directamente desde G ---
nodos_bomberos = [n for n, d in G.nodes(data=True) if d.get("tipo_nodo") == "parque_bomberos"]
nodos_hospitales = [n for n, d in G.nodes(data=True) if d.get("tipo_nodo") == "hospital"]

alturas = collections.Counter(d["height_m"] for _, _, d in G.edges(data=True))

alcanzabilidad = []
for n in nodos_bomberos:
    alcanzables = len(nx.descendants(G, n)) + 1
    alcanzabilidad.append(round(alcanzables / G.number_of_nodes() * 100, 1))

componentes = list(nx.weakly_connected_components(G))

stats = {
    "nodos_totales": G.number_of_nodes(),
    "aristas_totales": G.number_of_edges(),
    "nodos_parques_bomberos": len(nodos_bomberos),
    "nodos_hospitales": len(nodos_hospitales),
    "galibo_distribucion_m": {str(k): v for k, v in alturas.items()},
    "conectividad_componentes_debiles": len(componentes),
    "conectividad_alcance_medio_desde_parque_%": round(sum(alcanzabilidad)/len(alcanzabilidad), 1),
    "conectividad_alcance_min_%": min(alcanzabilidad),
    "conectividad_alcance_max_%": max(alcanzabilidad),
}

STATS_OUT = "../data/processed/estadisticas_grafo_p2.json"
with open(STATS_OUT, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f"\nEstadísticas guardadas en: {STATS_OUT}\n")
print(json.dumps(stats, indent=2, ensure_ascii=False))

Grafo guardado: ../data/processed/grafo_madrid.pkl (29.2 MB)

Estadísticas guardadas en: ../data/processed/estadisticas_grafo_p2.json

{
  "nodos_totales": 171422,
  "aristas_totales": 235144,
  "nodos_parques_bomberos": 13,
  "nodos_hospitales": 53,
  "galibo_distribucion_m": {
    "99.0": 228496,
    "3.5": 6375,
    "3.0": 273
  },
  "conectividad_componentes_debiles": 1,
  "conectividad_alcance_medio_desde_parque_%": 98.7,
  "conectividad_alcance_min_%": 98.7,
  "conectividad_alcance_max_%": 98.7
}
